### Imports

In [316]:
import os
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

### Inputs

In [317]:
edges_input_df = pd.read_csv("../inputs/edges_input.csv")
edges_input = edges_input_df[["rssi", "etx", "delay", "busy_fraction"]].values

In [318]:
radnet_input_df = pd.read_csv("../inputs/radnet_input.csv")
radnet_input_efsr_edge = radnet_input_df['efsr_edge'].values

### Outputs

In [319]:
inference_output_df = pd.read_csv("../outputs/inference_output.csv")
inference_output_class = inference_output_df['class'].values
inference_output_confidence = inference_output_df['confidence'].values

### Fuzzy Logic

In [320]:
conf_model = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'conf_model')
conf_efsr = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'conf_efsr')
quality = ctrl.Consequent(np.arange(0, 1.01, 0.01), 'quality')

In [321]:
conf_model['baixa'] = fuzz.trimf(conf_model.universe, [0, 0, 0.8])
conf_model['media'] = fuzz.trimf(conf_model.universe, [0.7, 0.85, 0.95])
conf_model['alta'] = fuzz.trimf(conf_model.universe, [0.9, 1, 1])

In [322]:
conf_efsr['baixa'] = fuzz.trimf(conf_efsr.universe, [0, 0, 0.5])
conf_efsr['media'] = fuzz.trimf(conf_efsr.universe, [0.4, 0.6, 0.8])
conf_efsr['alta'] = fuzz.trimf(conf_efsr.universe, [0.7, 1, 1])

In [323]:
quality['ruim'] = fuzz.trimf(quality.universe, [0, 0, 0.35])
quality['media'] = fuzz.trimf(quality.universe, [0.25, 0.5, 0.75])
quality['boa'] = fuzz.trimf(quality.universe, [0.65, 1, 1])

In [324]:
rules = [
    ctrl.Rule(conf_model['alta'] & conf_efsr['alta'], quality['boa']),
    ctrl.Rule(conf_model['media'] & conf_efsr['alta'], quality['boa']),
    ctrl.Rule(conf_model['baixa'] & conf_efsr['alta'], quality['boa']),
    ctrl.Rule(conf_model['alta'] & conf_efsr['media'], quality['boa']),
    ctrl.Rule(conf_model['media'] & conf_efsr['media'], quality['media']),
    ctrl.Rule(conf_model['baixa'] & conf_efsr['media'], quality['media']),
    ctrl.Rule(conf_model['alta'] & conf_efsr['baixa'], quality['media']),
    ctrl.Rule(conf_model['media'] & conf_efsr['baixa'], quality['ruim']),
    ctrl.Rule(conf_model['baixa'] & conf_efsr['baixa'], quality['ruim'])
]

In [325]:
fuzzy_system = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

In [326]:
# Normalização
confidence_efsr = np.clip((radnet_input_efsr_edge - 0.20) / (0.98 - 0.20), 0, 1) 

In [327]:
fuzzy = np.zeros(len(inference_output_confidence))

for i in range(len(inference_output_confidence)):
    fuzzy_system.input['conf_model'] = inference_output_confidence[i]
    fuzzy_system.input['conf_efsr'] = confidence_efsr[i]
    fuzzy_system.compute()
    fuzzy[i] = fuzzy_system.output['quality']

In [328]:
cls = inference_output_class
conf = fuzzy

In [329]:
print(f"{'RSSI':<6} {'ETX':<5} {'Delay':<6} {'Busy':<6} {'EFSR':<6} {'Class':<8} {'Status':<8} {'Confidence':<10}")
print(f"{'-'*62}")

CLASS_NAMES = {0: "Bad", 1: "Good"}

for i, row in enumerate(edges_input):
    print(f"{row[0]:<6.0f} {row[1]:<5.1f} {row[2]:<6.0f} {row[3]:<6.2f} "
          f"{radnet_input_efsr_edge[i]:<6.2f} "
          f"{cls[i]:<8} {CLASS_NAMES[int(cls[i])]:<8} {conf[i]:<10.2f}")

RSSI   ETX   Delay  Busy   EFSR   Class    Status   Confidence
--------------------------------------------------------------
-59    1.1   17     0.20   0.33   1        Good     0.50      
-60    1.4   6      0.10   0.91   1        Good     0.87      
-78    6.8   19     0.49   0.91   0        Bad      0.88      
-84    11.0  18     0.26   0.36   0        Bad      0.50      
-64    3.2   7      0.20   0.36   1        Good     0.50      
-43    1.5   4      0.32   0.90   1        Good     0.87      
-58    1.3   4      0.22   0.90   1        Good     0.87      
-57    1.2   14     0.17   0.90   1        Good     0.87      
-62    1.0   7      0.11   0.93   1        Good     0.88      
-69    1.8   5      0.22   0.38   1        Good     0.50      
-62    3.1   7      0.34   0.32   1        Good     0.50      
-57    1.0   14     0.30   0.35   1        Good     0.50      
-49    2.6   18     0.19   0.41   1        Good     0.50      
-78    4.4   42     0.82   0.48   0        Bad      0.5

### Inverter

In [330]:
# Guardar valores originais
cls_original = inference_output_class.copy()
conf_original = fuzzy.copy()

# Criar versão ajustada pelo fuzzy
cls_fuzzy = cls_original.copy()
conf_fuzzy = conf_original.copy()

# Limiar para inversão
limiar = 0.5

# Inverter classificação e confiança quando conf < limiar
for i in range(len(cls_fuzzy)):
    if conf_original[i] < limiar:
        cls_fuzzy[i] = 1 - cls_original[i]  # Inverte classe (0→1, 1→0)
        conf_fuzzy[i] = 1 - conf_original[i]  # Inverte confiança (0.3→0.7)

In [331]:
cls = cls_fuzzy
conf = conf_fuzzy

### Results

In [332]:
print(f"{'RSSI':<6} {'ETX':<5} {'Delay':<6} {'Busy':<6} {'EFSR':<6} {'Class':<8} {'Status':<8} {'Confidence':<10}")
print(f"{'-'*62}")

CLASS_NAMES = {0: "Bad", 1: "Good"}

for i, row in enumerate(edges_input):
    print(f"{row[0]:<6.0f} {row[1]:<5.1f} {row[2]:<6.0f} {row[3]:<6.2f} "
          f"{radnet_input_efsr_edge[i]:<6.2f} "
          f"{cls[i]:<8} {CLASS_NAMES[int(cls[i])]:<8} {conf[i]:<10.2f}")

RSSI   ETX   Delay  Busy   EFSR   Class    Status   Confidence
--------------------------------------------------------------
-59    1.1   17     0.20   0.33   1        Good     0.50      
-60    1.4   6      0.10   0.91   1        Good     0.87      
-78    6.8   19     0.49   0.91   0        Bad      0.88      
-84    11.0  18     0.26   0.36   0        Bad      0.50      
-64    3.2   7      0.20   0.36   0        Bad      0.50      
-43    1.5   4      0.32   0.90   1        Good     0.87      
-58    1.3   4      0.22   0.90   1        Good     0.87      
-57    1.2   14     0.17   0.90   1        Good     0.87      
-62    1.0   7      0.11   0.93   1        Good     0.88      
-69    1.8   5      0.22   0.38   0        Bad      0.50      
-62    3.1   7      0.34   0.32   0        Bad      0.50      
-57    1.0   14     0.30   0.35   1        Good     0.50      
-49    2.6   18     0.19   0.41   1        Good     0.50      
-78    4.4   42     0.82   0.48   0        Bad      0.5

### Save

In [333]:
fuzzy_output = pd.DataFrame({
    'class': cls,
    'confidence': conf
})

os.makedirs("../outputs", exist_ok=True)
fuzzy_output.to_csv("../outputs/fuzzy_output.csv", index=False)

print("Lógica Fuzzy salva em ../outputs/fuzzy_output.csv")

Lógica Fuzzy salva em ../outputs/fuzzy_output.csv
